In [5]:
"""
Lei and Masanet (2022) WUE Simulation Using REAL TMY Data
========================================================

This code downloads and uses the same TMY weather data files
that Lei and Masanet (2022) used for their simulations.

Source: EnergyPlus Weather Data (https://energyplus.net/weather)
Representative cities from DOE Reference Buildings by Climate Zone

REQUIRED FILES IN SAME DIRECTORY:
- simulation_funs_DC.py
- COP_2.pkl
- COP_AC.pkl
- COP_DX.pkl

Citation:
Lei, N., and Masanet, E. (2022). Climate and technology specific PUE and WUE
estimations for U.S. data centers. Resources, Conservation and Recycling, 182, 106323.
"""

import numpy as np
import pandas as pd
import requests
import os
import warnings
import pickle

warnings.filterwarnings("ignore")

from simulation_funs_DC import (
    PUE_WUE_AE_Chiller,    # Case 1 Hyperscale
    PUE_WUE_Chiller,       # Case 5 Traditional evaporative
    PUE_WUE_AIRChiller,    # Case 7 Air cooled
)

print("=" * 80)
print("LEI AND MASANET (2022) WUE SIMULATION")
print("Using REAL TMY Weather Data from EnergyPlus")
print("=" * 80)


# =============================================================================
# CLIMATE ZONE DEFINITIONS WITH TMY FILE URLS
# =============================================================================

CLIMATE_ZONES = {
    "1A": {
        "name": "Very Hot Humid",
        "city": "Miami",
        "state": "FL",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/FL/USA_FL_Miami.Intl.AP.722020_TMY3/USA_FL_Miami.Intl.AP.722020_TMY3.epw",
    },
    "2A": {
        "name": "Hot Humid",
        "city": "Houston",
        "state": "TX",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/TX/USA_TX_Houston-Bush.Intercontinental.AP.722430_TMY3/USA_TX_Houston-Bush.Intercontinental.AP.722430_TMY3.epw",
    },
    "2B": {
        "name": "Hot Dry",
        "city": "Phoenix",
        "state": "AZ",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/AZ/USA_AZ_Phoenix-Sky.Harbor.Intl.AP.722780_TMY3/USA_AZ_Phoenix-Sky.Harbor.Intl.AP.722780_TMY3.epw",
    },
    "3A": {
        "name": "Warm Humid",
        "city": "Atlanta",
        "state": "GA",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/GA/USA_GA_Atlanta-Hartsfield-Jackson.Intl.AP.722190_TMY3/USA_GA_Atlanta-Hartsfield-Jackson.Intl.AP.722190_TMY3.epw",
    },
    "3B": {
        "name": "Warm Dry",
        "city": "Las Vegas",
        "state": "NV",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/NV/USA_NV_Las.Vegas-McCarran.Intl.AP.723860_TMY3/USA_NV_Las.Vegas-McCarran.Intl.AP.723860_TMY3.epw",
    },
    "3C": {
        "name": "Warm Marine",
        "city": "San Francisco",
        "state": "CA",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/CA/USA_CA_San.Francisco.Intl.AP.724940_TMY3/USA_CA_San.Francisco.Intl.AP.724940_TMY3.epw",
    },
    "4A": {
        "name": "Mixed Humid",
        "city": "Baltimore",
        "state": "MD",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/MD/USA_MD_Baltimore-Washington.Intl.AP.724060_TMY3/USA_MD_Baltimore-Washington.Intl.AP.724060_TMY3.epw",
    },
    "4B": {
        "name": "Mixed Dry",
        "city": "Albuquerque",
        "state": "NM",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/NM/USA_NM_Albuquerque.Intl.AP.723650_TMY3/USA_NM_Albuquerque.Intl.AP.723650_TMY3.epw",
    },
    "4C": {
        "name": "Mixed Marine",
        "city": "Seattle",
        "state": "WA",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/WA/USA_WA_Seattle-Tacoma.Intl.AP.727930_TMY3/USA_WA_Seattle-Tacoma.Intl.AP.727930_TMY3.epw",
    },
    "5A": {
        "name": "Cool Humid",
        "city": "Chicago",
        "state": "IL",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/IL/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw",
    },
    "5B": {
        "name": "Cool Dry",
        "city": "Denver",
        "state": "CO",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/CO/USA_CO_Denver.Intl.AP.725650_TMY3/USA_CO_Denver.Intl.AP.725650_TMY3.epw",
    },
    "6A": {
        "name": "Cold Humid",
        "city": "Minneapolis",
        "state": "MN",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/MN/USA_MN_Minneapolis-St.Paul.Intl.AP.726580_TMY3/USA_MN_Minneapolis-St.Paul.Intl.AP.726580_TMY3.epw",
    },
    "6B": {
        "name": "Cold Dry",
        "city": "Helena",
        "state": "MT",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/MT/USA_MT_Helena.Rgnl.AP.727720_TMY3/USA_MT_Helena.Rgnl.AP.727720_TMY3.epw",
    },
    "7": {
        "name": "Very Cold",
        "city": "Duluth",
        "state": "MN",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/MN/USA_MN_Duluth.Intl.AP.727450_TMY3/USA_MN_Duluth.Intl.AP.727450_TMY3.epw",
    },
    "8": {
        "name": "Subarctic",
        "city": "Fairbanks",
        "state": "AK",
        "epw_url": "https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/USA/AK/USA_AK_Fairbanks.Intl.AP.702610_TMY3/USA_AK_Fairbanks.Intl.AP.702610_TMY3.epw",
    },
}


# =============================================================================
# DATA CENTER LOCATIONS MAPPED TO CLIMATE ZONES
# =============================================================================

DATA_CENTER_LOCATIONS = {
    "Phoenix_AZ": {
        "station_id": "09502000",
        "climate_zone": "2B",
        "water_source": "Salt River via SRP CAP",
        "match_quality": "EXACT same city as representative",
    },
    "Northern_Virginia": {
        "station_id": "01646500",
        "climate_zone": "4A",
        "water_source": "Potomac River",
        "match_quality": "EXCELLENT same zone near Baltimore",
    },
    "The_Dalles_OR": {
        "station_id": "14105700",
        "climate_zone": "5B",
        "water_source": "Columbia River",
        "match_quality": "GOOD same zone Columbia Gorge",
    },
    "Council_Bluffs_IA": {
        "station_id": "06610000",
        "climate_zone": "5A",
        "water_source": "Missouri River",
        "match_quality": "EXCELLENT same zone as Chicago",
    },
    "Prineville_OR": {
        "station_id": "14087400",
        "climate_zone": "5B",
        "water_source": "Crooked River",
        "match_quality": "GOOD high desert similar to Denver zone",
    },
    "Altoona_Des_Moines_IA": {
        "station_id": "05484500",
        "climate_zone": "5A",
        "water_source": "Raccoon Des Moines system",
        "match_quality": "EXCELLENT same zone as Chicago",
    },
    "Seattle_WA": {
        "station_id": "12117500",
        "climate_zone": "4C",
        "water_source": "Cedar River",
        "match_quality": "EXACT same city as representative",
    },
    "Quincy_WA": {
        "station_id": "12472800",
        "climate_zone": "5B",
        "water_source": "Columbia River below Priest Rapids Dam",
        "match_quality": "GOOD Columbia Basin",
    },
    "Dallas_Fort_Worth_TX": {
        "station_id": "08057000",
        "climate_zone": "3A",
        "water_source": "Trinity River",
        "match_quality": "GOOD same zone as Atlanta",
    },
    "San_Antonio_TX": {
        "station_id": "08178000",
        "climate_zone": "2A",
        "water_source": "San Antonio River and Edwards Aquifer",
        "match_quality": "EXCELLENT same zone as Houston",
    },
    "Atlanta_GA": {
        "station_id": "02336000",
        "climate_zone": "3A",
        "water_source": "Chattahoochee River",
        "match_quality": "EXACT same city as representative",
    },
    "Silicon_Valley_CA": {
        "station_id": "11169000",
        "climate_zone": "3C",
        "water_source": "Guadalupe River and imports",
        "match_quality": "EXACT same metro as San Francisco",
    },
    "Las_Vegas_NV": {
        "station_id": "09419800",
        "climate_zone": "3B",
        "water_source": "Colorado River via Lake Mead",
        "match_quality": "EXACT same city as representative",
    },
}


# =============================================================================
# EPW FILE DOWNLOAD AND PARSER
# =============================================================================

def download_epw_file(url, save_dir="TMY_Data"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    filename = url.split("/")[-1]
    filepath = os.path.join(save_dir, filename)

    if os.path.exists(filepath):
        print(f"    Using cached: {filename}")
        return filepath

    print(f"    Downloading: {filename}")
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(response.content)
        return filepath
    except Exception as e:
        print(f"    ERROR downloading {filename}: {e}")
        return None


def parse_epw_file(filepath):
    T_oa = []
    RH_oa = []
    P_oa = []

    with open(filepath, "r") as f:
        lines = f.readlines()

    for line in lines[8:]:
        parts = line.strip().split(",")
        if len(parts) >= 10:
            try:
                T = float(parts[6])
                RH = float(parts[8])
                P = float(parts[9])

                if -60 <= T <= 60 and 0 <= RH <= 100 and 50000 <= P <= 110000:
                    T_oa.append(T)
                    RH_oa.append(RH)
                    P_oa.append(P)
            except (ValueError, IndexError):
                continue

    return np.array(T_oa), np.array(RH_oa), np.array(P_oa)


# =============================================================================
# PARAMETER FUNCTIONS
# =============================================================================

def get_case1_params(T_oa, RH_oa, P_oa, practice="central"):
    params = {
        "best": {
            "UPS_e": 0.99, "PD_lr": 0.00, "L_percentage": 0.00,
            "delta_T_air": 19.4, "Fan_Pressure_CRAC": 300, "Fan_e_CRAC": 0.90,
            "Pump_Pressure_HD": 6300, "Pump_e_HD": 0.80, "AT_CT": 2.8,
            "Chiller_load": 0.8, "delta_T_water": 10, "Pump_Pressure_CW": 114900,
            "Pump_e_CW": 0.80, "delta_T_CT": 6, "Pump_Pressure_CT": 166900,
            "Pump_e_CT": 0.80, "Windage_p": 0.00005, "CC": 15,
            "Fan_Pressure_CT": 100, "Fan_e_CT": 0.90, "SHR": 0.99, "LGRatio": 4,
            "T_up": 35, "T_lw": 10, "dp_up": 27, "dp_lw": -12, "RH_up": 95, "RH_lw": 8,
            "pcop": 0.11,
        },
        "central": {
            "UPS_e": 0.945, "PD_lr": 0.01, "L_percentage": 0.001,
            "delta_T_air": 16.65, "Fan_Pressure_CRAC": 650, "Fan_e_CRAC": 0.775,
            "Pump_Pressure_HD": 7000, "Pump_e_HD": 0.70, "AT_CT": 4.75,
            "Chiller_load": 0.5, "delta_T_water": 7.5, "Pump_Pressure_CW": 143650,
            "Pump_e_CW": 0.70, "delta_T_CT": 5, "Pump_Pressure_CT": 208650,
            "Pump_e_CT": 0.70, "Windage_p": 0.0025, "CC": 9,
            "Fan_Pressure_CT": 250, "Fan_e_CT": 0.775, "SHR": 0.97, "LGRatio": 2.1,
            "T_up": 31, "T_lw": 14, "dp_up": 21, "dp_lw": -10.5, "RH_up": 77.5, "RH_lw": 14,
            "pcop": 0.0,
        },
        "poor": {
            "UPS_e": 0.90, "PD_lr": 0.02, "L_percentage": 0.002,
            "delta_T_air": 13.9, "Fan_Pressure_CRAC": 1000, "Fan_e_CRAC": 0.65,
            "Pump_Pressure_HD": 7700, "Pump_e_HD": 0.60, "AT_CT": 6.7,
            "Chiller_load": 0.2, "delta_T_water": 5, "Pump_Pressure_CW": 172400,
            "Pump_e_CW": 0.60, "delta_T_CT": 4, "Pump_Pressure_CT": 250400,
            "Pump_e_CT": 0.60, "Windage_p": 0.005, "CC": 3,
            "Fan_Pressure_CT": 400, "Fan_e_CT": 0.65, "SHR": 0.95, "LGRatio": 0.2,
            "T_up": 27, "T_lw": 18, "dp_up": 15, "dp_lw": -9, "RH_up": 60, "RH_lw": 20,
            "pcop": -0.11,
        },
    }
    p = params[practice]

    w = [
        T_oa, RH_oa, P_oa,
        p["UPS_e"], p["PD_lr"], p["L_percentage"], p["delta_T_air"],
        p["Fan_Pressure_CRAC"], p["Fan_e_CRAC"], p["Pump_Pressure_HD"], p["Pump_e_HD"],
        p["AT_CT"], p["Chiller_load"], p["delta_T_water"], p["Pump_Pressure_CW"],
        p["Pump_e_CW"], p["delta_T_CT"], p["Pump_Pressure_CT"], p["Pump_e_CT"],
        p["Windage_p"], p["CC"], p["Fan_Pressure_CT"], p["Fan_e_CT"],
        p["SHR"], p["LGRatio"], p["T_up"], p["T_lw"], p["dp_up"], p["dp_lw"],
        p["RH_up"], p["RH_lw"], p["pcop"],
    ]
    return w


def get_case5_params(T_oa, RH_oa, P_oa, practice="central"):
    params = {
        "best": {
            "UPS_e": 0.94, "PD_lr": 0.02, "L_percentage": 0.02,
            "SHR": 0.99, "delta_T_air": 10, "Fan_Pressure_CRAC": 400, "Fan_e_CRAC": 0.80,
            "Pump_Pressure_HD": 6300, "Pump_e_HD": 0.80, "HTE": 0.90,
            "delta_T_water": 10, "Pump_Pressure_CW": 114900, "Pump_e_CW": 0.80,
            "AT_CT": 2.8, "Chiller_load": 0.5,
            "delta_T_CT": 6, "Pump_Pressure_CT": 166900, "Pump_e_CT": 0.80,
            "Windage_p": 0.00005, "CC": 12, "LGRatio": 2.0,
            "Fan_Pressure_CT": 200, "Fan_e_CT": 0.80,
            "T_up": 32, "T_lw": 15, "dp_up": 27, "dp_lw": -12, "RH_up": 80, "RH_lw": 10,
            "pcop": 0.0,
        },
        "central": {
            "UPS_e": 0.87, "PD_lr": 0.035, "L_percentage": 0.035,
            "SHR": 0.97, "delta_T_air": 7.5, "Fan_Pressure_CRAC": 650, "Fan_e_CRAC": 0.70,
            "Pump_Pressure_HD": 7000, "Pump_e_HD": 0.70, "HTE": 0.775,
            "delta_T_water": 7.5, "Pump_Pressure_CW": 143650, "Pump_e_CW": 0.70,
            "AT_CT": 4.75, "Chiller_load": 0.3,
            "delta_T_CT": 5, "Pump_Pressure_CT": 208650, "Pump_e_CT": 0.70,
            "Windage_p": 0.0025, "CC": 7.5, "LGRatio": 1.1,
            "Fan_Pressure_CT": 300, "Fan_e_CT": 0.70,
            "T_up": 29.5, "T_lw": 16.5, "dp_up": 21, "dp_lw": -10.5, "RH_up": 70, "RH_lw": 20,
            "pcop": -0.2,
        },
        "poor": {
            "UPS_e": 0.80, "PD_lr": 0.05, "L_percentage": 0.05,
            "SHR": 0.95, "delta_T_air": 5, "Fan_Pressure_CRAC": 900, "Fan_e_CRAC": 0.60,
            "Pump_Pressure_HD": 7700, "Pump_e_HD": 0.60, "HTE": 0.65,
            "delta_T_water": 5, "Pump_Pressure_CW": 172400, "Pump_e_CW": 0.60,
            "AT_CT": 6.7, "Chiller_load": 0.1,
            "delta_T_CT": 4, "Pump_Pressure_CT": 250400, "Pump_e_CT": 0.60,
            "Windage_p": 0.005, "CC": 3, "LGRatio": 0.2,
            "Fan_Pressure_CT": 400, "Fan_e_CT": 0.60,
            "T_up": 27, "T_lw": 18, "dp_up": 15, "dp_lw": -9, "RH_up": 60, "RH_lw": 30,
            "pcop": -0.4,
        },
    }
    p = params[practice]

    w = [
        T_oa, RH_oa, P_oa,
        p["UPS_e"], p["PD_lr"], p["L_percentage"], p["SHR"], p["delta_T_air"],
        p["Fan_Pressure_CRAC"], p["Fan_e_CRAC"], p["Pump_Pressure_HD"], p["Pump_e_HD"],
        p["HTE"], p["delta_T_water"], p["Pump_Pressure_CW"], p["Pump_e_CW"],
        p["AT_CT"], p["Chiller_load"],
        p["delta_T_CT"], p["Pump_Pressure_CT"], p["Pump_e_CT"],
        p["Windage_p"], p["CC"], p["Fan_Pressure_CT"], p["Fan_e_CT"],
        p["LGRatio"],
        p["T_up"], p["T_lw"], p["dp_up"], p["dp_lw"], p["RH_up"], p["RH_lw"], p["pcop"],
    ]
    return w


def get_case7_params(T_oa, RH_oa, P_oa, practice="central"):
    params = {
        "best": {
            "UPS_e": 0.94, "PD_lr": 0.02, "L_percentage": 0.02,
            "SHR": 0.99, "delta_T_air": 10, "Fan_Pressure_CRAC": 400, "Fan_e_CRAC": 0.80,
            "Pump_Pressure_HD": 6300, "Pump_e_HD": 0.80, "HTE": 0.90,
            "delta_T_water": 10, "Pump_Pressure_CW": 114900, "Pump_e_CW": 0.80,
            "Chiller_load": 0.5, "pcop": -0.25,
            "T_up": 32, "T_lw": 15, "dp_up": 27, "dp_lw": -12, "RH_up": 80, "RH_lw": 10,
        },
        "central": {
            "UPS_e": 0.87, "PD_lr": 0.035, "L_percentage": 0.035,
            "SHR": 0.97, "delta_T_air": 7.5, "Fan_Pressure_CRAC": 650, "Fan_e_CRAC": 0.70,
            "Pump_Pressure_HD": 7000, "Pump_e_HD": 0.70, "HTE": 0.775,
            "delta_T_water": 7.5, "Pump_Pressure_CW": 143650, "Pump_e_CW": 0.70,
            "Chiller_load": 0.3, "pcop": -0.325,
            "T_up": 29.5, "T_lw": 16.5, "dp_up": 21, "dp_lw": -10.5, "RH_up": 70, "RH_lw": 20,
        },
        "poor": {
            "UPS_e": 0.80, "PD_lr": 0.05, "L_percentage": 0.05,
            "SHR": 0.95, "delta_T_air": 5, "Fan_Pressure_CRAC": 900, "Fan_e_CRAC": 0.60,
            "Pump_Pressure_HD": 7700, "Pump_e_HD": 0.60, "HTE": 0.65,
            "delta_T_water": 5, "Pump_Pressure_CW": 172400, "Pump_e_CW": 0.60,
            "Chiller_load": 0.1, "pcop": -0.40,
            "T_up": 27, "T_lw": 18, "dp_up": 15, "dp_lw": -9, "RH_up": 60, "RH_lw": 30,
        },
    }
    p = params[practice]

    w = [
        T_oa, RH_oa, P_oa,
        p["UPS_e"], p["PD_lr"], p["L_percentage"], p["SHR"], p["delta_T_air"],
        p["Fan_Pressure_CRAC"], p["Fan_e_CRAC"], p["Pump_Pressure_HD"], p["Pump_e_HD"],
        p["HTE"], p["delta_T_water"], p["Pump_Pressure_CW"], p["Pump_e_CW"],
        p["Chiller_load"], p["pcop"],
        p["T_up"], p["T_lw"], p["dp_up"], p["dp_lw"], p["RH_up"], p["RH_lw"],
    ]
    return w


# =============================================================================
# SIMULATION RUNNER
# =============================================================================

def run_zone_simulation(zone_code, T_oa, RH_oa, P_oa, sample_interval=1):
    T = T_oa[::sample_interval]
    RH = RH_oa[::sample_interval]
    P = P_oa[::sample_interval]

    results = {}

    cases = [
        ("Case1_Hyperscale", PUE_WUE_AE_Chiller, get_case1_params),
        ("Case5_Evaporative", PUE_WUE_Chiller, get_case5_params),
        ("Case7_AirCooled", PUE_WUE_AIRChiller, get_case7_params),
    ]

    for case_name, case_func, param_func in cases:
        for practice in ["best", "central", "poor"]:
            pue_list = []
            wue_list = []

            for i in range(len(T)):
                try:
                    w = param_func(T[i], RH[i], P[i], practice)
                    pue, wue = case_func(w)
                    if (
                        not np.isnan(pue)
                        and not np.isnan(wue)
                        and wue >= 0
                        and pue > 1
                    ):
                        pue_list.append(pue)
                        wue_list.append(wue)
                except Exception:
                    pass

            key = f"{case_name}_{practice}"
            if len(wue_list) > 100:
                results[key] = {
                    "PUE_mean": float(np.mean(pue_list)),
                    "WUE_mean": float(np.mean(wue_list)),
                    "WUE_min": float(np.min(wue_list)),
                    "WUE_max": float(np.max(wue_list)),
                    "n_valid_hours": int(len(wue_list)),
                    "n_total_hours": int(len(T)),
                }
            else:
                results[key] = {
                    "PUE_mean": np.nan,
                    "WUE_mean": np.nan,
                    "n_valid_hours": int(len(wue_list)),
                    "n_total_hours": int(len(T)),
                }

    return results


# =============================================================================
# STEP 1 DOWNLOAD TMY DATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1 DOWNLOAD TMY WEATHER DATA")
print("=" * 80)

zone_tmy_data = {}
zone_results = {}

for zone_code, zone_info in CLIMATE_ZONES.items():
    print(f"\nZone {zone_code}: {zone_info['name']} ({zone_info['city']}, {zone_info['state']})")

    epw_path = download_epw_file(zone_info["epw_url"])

    if epw_path:
        T_oa, RH_oa, P_oa = parse_epw_file(epw_path)

        if len(T_oa) > 8000:
            zone_tmy_data[zone_code] = {
                "T_oa": T_oa,
                "RH_oa": RH_oa,
                "P_oa": P_oa,
                "T_mean": float(np.mean(T_oa)),
                "T_min": float(np.min(T_oa)),
                "T_max": float(np.max(T_oa)),
                "RH_mean": float(np.mean(RH_oa)),
                "P_mean": float(np.mean(P_oa)),
            }
            print(
                f"    Loaded {len(T_oa)} hours "
                f"Tavg = {np.mean(T_oa):.1f} C  RHavg = {np.mean(RH_oa):.0f} percent"
            )
        else:
            print(f"    Insufficient data: {len(T_oa)} hours")
    else:
        print("    Failed to download EPW file")


# =============================================================================
# STEP 2 RUN SIMULATIONS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2 RUNNING SIMULATIONS")
print("=" * 80)

for zone_code, tmy in zone_tmy_data.items():
    zone_info = CLIMATE_ZONES[zone_code]

    print(f"\nSimulating Zone {zone_code}: {zone_info['city']}")

    zone_results[zone_code] = run_zone_simulation(
        zone_code,
        tmy["T_oa"],
        tmy["RH_oa"],
        tmy["P_oa"],
        sample_interval=1,
    )

    r = zone_results[zone_code]
    c1 = r.get("Case1_Hyperscale_central", {}).get("WUE_mean", np.nan)
    c5 = r.get("Case5_Evaporative_central", {}).get("WUE_mean", np.nan)
    print(
        f"    Hyperscale WUE central: {c1:.3f} L per kWh"
        if not np.isnan(c1)
        else "    Hyperscale central: N/A"
    )
    print(
        f"    Evaporative WUE central: {c5:.3f} L per kWh"
        if not np.isnan(c5)
        else "    Evaporative central: N/A"
    )

print("\nSimulations complete")


# =============================================================================
# TABLE 1 CLIMATE ZONE WUE RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("TABLE 1 CLIMATE ZONE WUE RESULTS")
print("=" * 80)

table1_data = []
for zone_code, r in zone_results.items():
    zone_info = CLIMATE_ZONES[zone_code]
    tmy = zone_tmy_data.get(zone_code, {})

    row = {
        "Zone": zone_code,
        "Climate_Type": zone_info["name"],
        "Representative_City": f"{zone_info['city']}, {zone_info['state']}",
        "T_avg_C": round(tmy.get("T_mean", np.nan), 1),
        "RH_avg_pct": round(tmy.get("RH_mean", np.nan), 0),
        "Hyperscale_Best": round(r.get("Case1_Hyperscale_best", {}).get("WUE_mean", np.nan), 3),
        "Hyperscale_Central": round(r.get("Case1_Hyperscale_central", {}).get("WUE_mean", np.nan), 3),
        "Hyperscale_Poor": round(r.get("Case1_Hyperscale_poor", {}).get("WUE_mean", np.nan), 3),
        "Evaporative_Best": round(r.get("Case5_Evaporative_best", {}).get("WUE_mean", np.nan), 3),
        "Evaporative_Central": round(r.get("Case5_Evaporative_central", {}).get("WUE_mean", np.nan), 3),
        "Evaporative_Poor": round(r.get("Case5_Evaporative_poor", {}).get("WUE_mean", np.nan), 3),
        "AirCooled_Central": round(r.get("Case7_AirCooled_central", {}).get("WUE_mean", np.nan), 3),
    }
    table1_data.append(row)

df_zones = pd.DataFrame(table1_data)
print(df_zones.to_string(index=False))


# =============================================================================
# TABLE 2 DATA CENTER LOCATIONS WITH WUE RANGES
# =============================================================================

print("\n" + "=" * 80)
print("TABLE 2 DATA CENTER LOCATIONS WITH WUE RANGES")
print("=" * 80)


def get_wue_range(r, case_prefix):
    vals = []
    for practice in ["best", "central", "poor"]:
        key = f"{case_prefix}_{practice}"
        w = r.get(key, {}).get("WUE_mean", np.nan)
        if w is not None and not np.isnan(w):
            vals.append(float(w))

    if not vals:
        return np.nan, np.nan, np.nan, "N/A"

    w_min = float(np.min(vals))
    w_max = float(np.max(vals))
    w_mid = 0.5 * (w_min + w_max)
    range_str = f"{w_min:.2f} - {w_max:.2f}"
    return w_min, w_max, w_mid, range_str


table2_data = []
pretty_rows = []

for loc_name, loc_info in DATA_CENTER_LOCATIONS.items():
    zone_code = loc_info["climate_zone"]
    r = zone_results.get(zone_code, {})
    zone_info = CLIMATE_ZONES.get(zone_code, {})

    c1_min, c1_max, c1_mid, c1_str = get_wue_range(r, "Case1_Hyperscale")
    c5_min, c5_max, c5_mid, c5_str = get_wue_range(r, "Case5_Evaporative")
    c7_min, c7_max, c7_mid, c7_str = get_wue_range(r, "Case7_AirCooled")

    row = {
        "Location": loc_name,
        "USGS_Station": loc_info["station_id"],
        "Climate_Zone": zone_code,
        "Climate_Type": zone_info.get("name", ""),
        "Representative_City": f"{zone_info.get('city', '')}, {zone_info.get('state', '')}",
        "Match_Quality": loc_info["match_quality"],
        "Water_Source": loc_info["water_source"],
        "WUE_TradEvap_min_LperkWh": round(c5_min, 3) if not np.isnan(c5_min) else np.nan,
        "WUE_TradEvap_max_LperkWh": round(c5_max, 3) if not np.isnan(c5_max) else np.nan,
        "WUE_TradEvap_mid_LperkWh": round(c5_mid, 3) if not np.isnan(c5_mid) else np.nan,
        "TradEvap_Range_LperkWh": c5_str,
        "WUE_Hyperscale_min_LperkWh": round(c1_min, 3) if not np.isnan(c1_min) else np.nan,
        "WUE_Hyperscale_max_LperkWh": round(c1_max, 3) if not np.isnan(c1_max) else np.nan,
        "WUE_Hyperscale_mid_LperkWh": round(c1_mid, 3) if not np.isnan(c1_mid) else np.nan,
        "Hyperscale_Range_LperkWh": c1_str,
        "WUE_AirCooled_min_LperkWh": round(c7_min, 3) if not np.isnan(c7_min) else np.nan,
        "WUE_AirCooled_max_LperkWh": round(c7_max, 3) if not np.isnan(c7_max) else np.nan,
        "WUE_AirCooled_mid_LperkWh": round(c7_mid, 3) if not np.isnan(c7_mid) else np.nan,
        "AirCooled_Range_LperkWh": c7_str,
    }
    table2_data.append(row)

    pretty_rows.append(
        {
            "Location": loc_name,
            "Traditional_Evaporative": c5_str,
            "Hyperscale_with_Economizer": c1_str,
            "Air_Cooled_Only": c7_str,
        }
    )

df_locations = pd.DataFrame(table2_data)
print("\nFULL NUMERIC TABLE FOR 13 LOCATIONS")
print(df_locations.to_string(index=False))

df_pretty = pd.DataFrame(pretty_rows)
print("\nRANGE TABLE SIMILAR TO CLIMATE TYPE FIGURE")
print(df_pretty.to_string(index=False))


# =============================================================================
# TABLE 3 PUE RESULTS BY CLIMATE ZONE
# =============================================================================

print("\n" + "=" * 80)
print("TABLE 3 PUE RESULTS BY CLIMATE ZONE")
print("=" * 80)

table3_data = []
for zone_code, r in zone_results.items():
    zone_info = CLIMATE_ZONES[zone_code]

    row = {
        "Zone": zone_code,
        "Climate_Type": zone_info["name"],
        "City": zone_info["city"],
        "Hyperscale_PUE_Best": round(r.get("Case1_Hyperscale_best", {}).get("PUE_mean", np.nan), 3),
        "Hyperscale_PUE_Central": round(r.get("Case1_Hyperscale_central", {}).get("PUE_mean", np.nan), 3),
        "Hyperscale_PUE_Poor": round(r.get("Case1_Hyperscale_poor", {}).get("PUE_mean", np.nan), 3),
        "Evaporative_PUE_Best": round(r.get("Case5_Evaporative_best", {}).get("PUE_mean", np.nan), 3),
        "Evaporative_PUE_Central": round(r.get("Case5_Evaporative_central", {}).get("PUE_mean", np.nan), 3),
        "Evaporative_PUE_Poor": round(r.get("Case5_Evaporative_poor", {}).get("PUE_mean", np.nan), 3),
    }
    table3_data.append(row)

df_pue = pd.DataFrame(table3_data)
print(df_pue.to_string(index=False))


# =============================================================================
# EXPORT TO CSV AND PKL
# =============================================================================

print("\n" + "=" * 80)
print("EXPORTING RESULTS")
print("=" * 80)

df_zones.to_csv("Lei_Masanet_WUE_By_ClimateZone.csv", index=False)
print("  Saved Lei_Masanet_WUE_By_ClimateZone.csv")

df_locations.to_csv("Lei_Masanet_WUE_13_DataCenter_Locations.csv", index=False)
print("  Saved Lei_Masanet_WUE_13_DataCenter_Locations.csv")

df_pretty.to_csv("Lei_Masanet_WUE_13_DataCenter_Ranges_Pretty.csv", index=False)
print("  Saved Lei_Masanet_WUE_13_DataCenter_Ranges_Pretty.csv")

df_pue.to_csv("Lei_Masanet_PUE_By_ClimateZone.csv", index=False)
print("  Saved Lei_Masanet_PUE_By_ClimateZone.csv")

with open("Lei_Masanet_Simulation_Results.pkl", "wb") as f:
    pickle.dump(
        {
            "zone_results": zone_results,
            "zone_tmy_data": {
                k: {key: val for key, val in v.items() if key not in ["T_oa", "RH_oa", "P_oa"]}
                for k, v in zone_tmy_data.items()
            },
            "climate_zones": CLIMATE_ZONES,
            "dc_locations": DATA_CENTER_LOCATIONS,
        },
        f,
    )
print("  Saved Lei_Masanet_Simulation_Results.pkl")

tmy_summary = []
for zone_code, tmy in zone_tmy_data.items():
    zone_info = CLIMATE_ZONES[zone_code]
    tmy_summary.append(
        {
            "Zone": zone_code,
            "City": zone_info["city"],
            "State": zone_info["state"],
            "T_mean_C": round(tmy["T_mean"], 1),
            "T_min_C": round(tmy["T_min"], 1),
            "T_max_C": round(tmy["T_max"], 1),
            "RH_mean_pct": round(tmy["RH_mean"], 0),
            "P_mean_Pa": round(tmy["P_mean"], 0),
        }
    )
df_tmy = pd.DataFrame(tmy_summary)
df_tmy.to_csv("TMY_Climate_Summary_By_Zone.csv", index=False)
print("  Saved TMY_Climate_Summary_By_Zone.csv")


# =============================================================================
# METHODOLOGY NOTE
# =============================================================================

print("\n" + "=" * 80)
print("METHODOLOGY NOTE")
print("=" * 80)

methodology_text = """
Source: Lei and Masanet (2022) Climate and technology specific PUE and WUE
estimations for U.S. data centers.

This script uses their published thermodynamic functions and COP models with
TMY3 weather data for 15 ASHRAE climate zones. For each zone it simulates
three cooling archetypes (hyperscale with economizer, traditional evaporative,
air cooled) under best, central, and poor practice parameter sets. For each
data center region the script maps to the corresponding climate zone and
builds WUE ranges (min, max, mid) for each cooling type. These ranges can be
used as inputs to basin level hydrological carrying capacity analysis.
"""

print(methodology_text)

with open("WUE_Methodology_Documentation.txt", "w") as f:
    f.write(methodology_text)
print("  Saved WUE_Methodology_Documentation.txt")

print("\n" + "=" * 80)
print("ALL EXPORTS COMPLETE")
print("=" * 80)


LEI AND MASANET (2022) WUE SIMULATION
Using REAL TMY Weather Data from EnergyPlus

STEP 1 DOWNLOAD TMY WEATHER DATA

Zone 1A: Very Hot Humid (Miami, FL)
    Using cached: USA_FL_Miami.Intl.AP.722020_TMY3.epw
    Loaded 8760 hours Tavg = 24.5 C  RHavg = 73 percent

Zone 2A: Hot Humid (Houston, TX)
    Using cached: USA_TX_Houston-Bush.Intercontinental.AP.722430_TMY3.epw
    Loaded 8760 hours Tavg = 20.4 C  RHavg = 73 percent

Zone 2B: Hot Dry (Phoenix, AZ)
    Using cached: USA_AZ_Phoenix-Sky.Harbor.Intl.AP.722780_TMY3.epw
    Loaded 8760 hours Tavg = 23.8 C  RHavg = 34 percent

Zone 3A: Warm Humid (Atlanta, GA)
    Using cached: USA_GA_Atlanta-Hartsfield-Jackson.Intl.AP.722190_TMY3.epw
    Loaded 8760 hours Tavg = 16.7 C  RHavg = 66 percent

Zone 3B: Warm Dry (Las Vegas, NV)
    Using cached: USA_NV_Las.Vegas-McCarran.Intl.AP.723860_TMY3.epw
    Loaded 8760 hours Tavg = 19.8 C  RHavg = 30 percent

Zone 3C: Warm Marine (San Francisco, CA)
    Using cached: USA_CA_San.Francisco.Intl.AP.7

In [ ]:
#!/usr/bin/env python3
"""
WUE Lookup Table Generator
Based on Lei & Masanet (2022) Published Ranges

This script creates WUE values (low, mid, high) for data center locations
using the published ranges from:

Lei, N., & Masanet, E. (2022). Climate- and technology-specific PUE and WUE 
estimations for U.S. data centers using a hybrid statistical and thermodynamics-
based approach. Resources, Conservation & Recycling, 182, 106323.

Methodology:
- WUE_low = lower end of published range (best practice)
- WUE_high = upper end of published range (poor practice)  
- WUE_mid = (WUE_low + WUE_high) / 2 (central estimate)

For air-cooled systems with "< X" notation:
- WUE_high = X (the stated upper bound)
- WUE_mid = X / 2
- WUE_low = X / 4
"""

import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple

# =============================================================================
# SECTION 1: DEFINE CLIMATE TYPES WITH WUE RANGES
# =============================================================================

# Lei & Masanet aggregate 15 ASHRAE zones into 7 climate types for WUE reporting
# Each entry: (WUE_low, WUE_high) for the range, or just upper bound for "< X"

@dataclass
class WUE_Range:
    """WUE range for a cooling type"""
    low: float
    high: float
    mid: float = None
    
    def __post_init__(self):
        if self.mid is None:
            self.mid = round((self.low + self.high) / 2, 4)  # Round to avoid floating point issues

@dataclass  
class ClimateWUE:
    """WUE values for all three cooling types in a climate"""
    traditional_evaporative: WUE_Range
    hyperscale_economizer: WUE_Range
    air_cooled: WUE_Range

# Define the 7 climate types with their WUE ranges from Lei & Masanet (2022)
CLIMATE_TYPE_WUE: Dict[str, ClimateWUE] = {
    
    "Hot-Dry": ClimateWUE(
        # Phoenix, Las Vegas - ASHRAE 2B, 3B
        traditional_evaporative = WUE_Range(low=2.3, high=3.9),
        hyperscale_economizer = WUE_Range(low=1.0, high=1.8),
        air_cooled = WUE_Range(low=0.025, high=0.1)  # "< 0.1"
    ),
    
    "Hot-Humid": ClimateWUE(
        # Miami, Houston - ASHRAE 1A, 2A
        traditional_evaporative = WUE_Range(low=2.5, high=4.0),
        hyperscale_economizer = WUE_Range(low=1.1, high=1.9),
        air_cooled = WUE_Range(low=0.025, high=0.1)  # "< 0.1"
    ),
    
    "Warm": ClimateWUE(
        # Atlanta, Dallas - ASHRAE 3A
        traditional_evaporative = WUE_Range(low=2.2, high=3.8),
        hyperscale_economizer = WUE_Range(low=0.8, high=1.6),
        air_cooled = WUE_Range(low=0.025, high=0.1)  # "< 0.1"
    ),
    
    "Marine": ClimateWUE(
        # Seattle, San Francisco - ASHRAE 3C, 4C
        traditional_evaporative = WUE_Range(low=2.0, high=3.5),
        hyperscale_economizer = WUE_Range(low=0.5, high=1.2),
        air_cooled = WUE_Range(low=0.0125, high=0.05)  # "< 0.05"
    ),
    
    "Mixed": ClimateWUE(
        # Northern Virginia - ASHRAE 4A
        traditional_evaporative = WUE_Range(low=2.1, high=3.7),
        hyperscale_economizer = WUE_Range(low=0.7, high=1.5),
        air_cooled = WUE_Range(low=0.02, high=0.08)  # "< 0.08"
    ),
    
    "Cool": ClimateWUE(
        # Chicago, Denver, Council Bluffs, Des Moines - ASHRAE 5A, 5B
        traditional_evaporative = WUE_Range(low=2.1, high=3.6),
        hyperscale_economizer = WUE_Range(low=0.6, high=1.3),
        air_cooled = WUE_Range(low=0.0125, high=0.05)  # "< 0.05"
    ),
    
    "Cold": ClimateWUE(
        # Minneapolis - ASHRAE 6A, 6B, 7
        traditional_evaporative = WUE_Range(low=1.9, high=3.4),
        hyperscale_economizer = WUE_Range(low=0.4, high=1.0),
        air_cooled = WUE_Range(low=0.0075, high=0.03)  # "< 0.03"
    ),
    
    "Subarctic": ClimateWUE(
        # Fairbanks - ASHRAE 8
        traditional_evaporative = WUE_Range(low=1.7, high=3.2),
        hyperscale_economizer = WUE_Range(low=0.2, high=0.6),
        air_cooled = WUE_Range(low=0.0075, high=0.03)  # "< 0.03"
    ),
}


# =============================================================================
# SECTION 2: DEFINE ALL 15 ASHRAE CLIMATE ZONES
# =============================================================================

# Map each ASHRAE zone to its climate type and representative city
ASHRAE_ZONES: Dict[str, Dict] = {
    # Zone: {climate_type, representative_city, description}
    
    "1A": {
        "climate_type": "Hot-Humid",
        "representative_city": "Miami, FL",
        "description": "Very Hot-Humid"
    },
    "2A": {
        "climate_type": "Hot-Humid",
        "representative_city": "Houston, TX",
        "description": "Hot-Humid"
    },
    "2B": {
        "climate_type": "Hot-Dry",
        "representative_city": "Phoenix, AZ",
        "description": "Hot-Dry"
    },
    "3A": {
        "climate_type": "Warm",
        "representative_city": "Atlanta, GA",
        "description": "Warm-Humid"
    },
    "3B": {
        "climate_type": "Hot-Dry",  # Las Vegas grouped with Hot-Dry
        "representative_city": "Las Vegas, NV",
        "description": "Warm-Dry"
    },
    "3C": {
        "climate_type": "Marine",
        "representative_city": "San Francisco, CA",
        "description": "Warm-Marine"
    },
    "4A": {
        "climate_type": "Mixed",
        "representative_city": "Baltimore, MD",
        "description": "Mixed-Humid"
    },
    "4B": {
        "climate_type": "Mixed",
        "representative_city": "Albuquerque, NM",
        "description": "Mixed-Dry"
    },
    "4C": {
        "climate_type": "Marine",
        "representative_city": "Seattle, WA",
        "description": "Mixed-Marine"
    },
    "5A": {
        "climate_type": "Cool",
        "representative_city": "Chicago, IL",
        "description": "Cool-Humid"
    },
    "5B": {
        "climate_type": "Cool",
        "representative_city": "Denver, CO",
        "description": "Cool-Dry"
    },
    "6A": {
        "climate_type": "Cold",
        "representative_city": "Minneapolis, MN",
        "description": "Cold-Humid"
    },
    "6B": {
        "climate_type": "Cold",
        "representative_city": "Helena, MT",
        "description": "Cold-Dry"
    },
    "7": {
        "climate_type": "Cold",
        "representative_city": "Duluth, MN",
        "description": "Very Cold"
    },
    "8": {
        "climate_type": "Subarctic",
        "representative_city": "Fairbanks, AK",
        "description": "Subarctic"
    },
}


# =============================================================================
# SECTION 3: DEFINE 13 DATA CENTER LOCATIONS
# =============================================================================

# Map each data center location to its ASHRAE zone, USGS station, and water source
DATA_CENTER_LOCATIONS: Dict[str, Dict] = {
    
    "Phoenix_AZ": {
        "ashrae_zone": "2B",
        "usgs_station": "09502000",
        "water_source": "Salt River (Verde confluence)",
        "match_quality": "Exact"
    },
    
    "Las_Vegas_NV": {
        "ashrae_zone": "3B",
        "usgs_station": "09419800",
        "water_source": "Las Vegas Wash / Lake Mead",
        "match_quality": "Exact"
    },
    
    "San_Antonio_TX": {
        "ashrae_zone": "2A",
        "usgs_station": "08178000",
        "water_source": "San Antonio River",
        "match_quality": "Exact"
    },
    
    "Dallas_Fort_Worth_TX": {
        "ashrae_zone": "3A",
        "usgs_station": "08057000",
        "water_source": "Trinity River",
        "match_quality": "Exact"
    },
    
    "Atlanta_GA": {
        "ashrae_zone": "3A",
        "usgs_station": "02336000",
        "water_source": "Chattahoochee River",
        "match_quality": "Exact"
    },
    
    "Northern_Virginia": {
        "ashrae_zone": "4A",
        "usgs_station": "01646500",
        "water_source": "Potomac River",
        "match_quality": "Exact"
    },
    
    "Silicon_Valley_CA": {
        "ashrae_zone": "3C",
        "usgs_station": "11169000",
        "water_source": "Guadalupe River",
        "match_quality": "Regional proxy"
    },
    
    "Seattle_WA": {
        "ashrae_zone": "4C",
        "usgs_station": "12113000",
        "water_source": "Green River",
        "match_quality": "Regional proxy"
    },
    
    "The_Dalles_OR": {
        "ashrae_zone": "5B",
        "usgs_station": "14105700",
        "water_source": "Columbia River",
        "match_quality": "Exact"
    },
    
    "Prineville_OR": {
        "ashrae_zone": "5B",
        "usgs_station": "14080500",
        "water_source": "Crooked River",
        "match_quality": "Exact"
    },
    
    "Quincy_WA": {
        "ashrae_zone": "5B",
        "usgs_station": "12472800",
        "water_source": "Columbia River (below Priest Rapids)",
        "match_quality": "Exact"
    },
    
    "Council_Bluffs_IA": {
        "ashrae_zone": "5A",
        "usgs_station": "06610000",
        "water_source": "Missouri River",
        "match_quality": "Exact"
    },
    
    "Des_Moines_IA": {
        "ashrae_zone": "5A",
        "usgs_station": "05481650",
        "water_source": "Des Moines River",
        "match_quality": "Exact"
    },
}


# =============================================================================
# SECTION 4: FUNCTIONS TO GET WUE VALUES
# =============================================================================

def get_wue_by_zone(ashrae_zone: str) -> ClimateWUE:
    """Get WUE values for an ASHRAE zone"""
    if ashrae_zone not in ASHRAE_ZONES:
        raise ValueError(f"Unknown ASHRAE zone: {ashrae_zone}")
    
    climate_type = ASHRAE_ZONES[ashrae_zone]["climate_type"]
    return CLIMATE_TYPE_WUE[climate_type]


def get_wue_by_location(location: str) -> Tuple[str, str, ClimateWUE]:
    """Get WUE values for a data center location
    
    Returns: (ashrae_zone, climate_type, ClimateWUE)
    """
    if location not in DATA_CENTER_LOCATIONS:
        raise ValueError(f"Unknown location: {location}")
    
    ashrae_zone = DATA_CENTER_LOCATIONS[location]["ashrae_zone"]
    climate_type = ASHRAE_ZONES[ashrae_zone]["climate_type"]
    wue = CLIMATE_TYPE_WUE[climate_type]
    
    return ashrae_zone, climate_type, wue


# =============================================================================
# SECTION 5: GENERATE OUTPUT TABLES
# =============================================================================

def generate_climate_type_table() -> pd.DataFrame:
    """Generate table of WUE ranges by climate type (like the image)"""
    
    rows = []
    for climate_type, wue in CLIMATE_TYPE_WUE.items():
        rows.append({
            "Climate_Type": climate_type,
            # Traditional Evaporative
            "Trad_Evap_Range": f"{wue.traditional_evaporative.low:.1f} - {wue.traditional_evaporative.high:.1f}",
            "Trad_Evap_Low": wue.traditional_evaporative.low,
            "Trad_Evap_Mid": wue.traditional_evaporative.mid,
            "Trad_Evap_High": wue.traditional_evaporative.high,
            # Hyperscale with Economizer
            "Hyperscale_Range": f"{wue.hyperscale_economizer.low:.1f} - {wue.hyperscale_economizer.high:.1f}",
            "Hyperscale_Low": wue.hyperscale_economizer.low,
            "Hyperscale_Mid": wue.hyperscale_economizer.mid,
            "Hyperscale_High": wue.hyperscale_economizer.high,
            # Air-Cooled
            "AirCooled_Range": f"< {wue.air_cooled.high:.2f}".replace("< 0.", "< ."),
            "AirCooled_Low": wue.air_cooled.low,
            "AirCooled_Mid": wue.air_cooled.mid,
            "AirCooled_High": wue.air_cooled.high,
        })
    
    return pd.DataFrame(rows)


def generate_location_wue_table() -> pd.DataFrame:
    """Generate WUE table for all 13 data center locations"""
    
    rows = []
    for location, info in DATA_CENTER_LOCATIONS.items():
        ashrae_zone, climate_type, wue = get_wue_by_location(location)
        
        rows.append({
            "Location": location,
            "USGS_Station": info["usgs_station"],
            "ASHRAE_Zone": ashrae_zone,
            "Climate_Type": climate_type,
            "Match_Quality": info["match_quality"],
            "Water_Source": info["water_source"],
            # Traditional Evaporative (TradEvap)
            "WUE_TradEvap_min_LperkWh": wue.traditional_evaporative.low,
            "WUE_TradEvap_mid_LperkWh": wue.traditional_evaporative.mid,
            "WUE_TradEvap_max_LperkWh": wue.traditional_evaporative.high,
            # Hyperscale with Economizer
            "WUE_Hyperscale_min_LperkWh": wue.hyperscale_economizer.low,
            "WUE_Hyperscale_mid_LperkWh": wue.hyperscale_economizer.mid,
            "WUE_Hyperscale_max_LperkWh": wue.hyperscale_economizer.high,
            # Air-Cooled
            "WUE_AirCooled_min_LperkWh": wue.air_cooled.low,
            "WUE_AirCooled_mid_LperkWh": wue.air_cooled.mid,
            "WUE_AirCooled_max_LperkWh": wue.air_cooled.high,
        })
    
    return pd.DataFrame(rows)


def generate_ashrae_zone_table() -> pd.DataFrame:
    """Generate reference table of all 15 ASHRAE zones"""
    
    rows = []
    for zone, info in ASHRAE_ZONES.items():
        wue = CLIMATE_TYPE_WUE[info["climate_type"]]
        
        rows.append({
            "ASHRAE_Zone": zone,
            "Description": info["description"],
            "Representative_City": info["representative_city"],
            "Climate_Type": info["climate_type"],
            # Just show hyperscale range as example
            "Hyperscale_WUE_Range": f"{wue.hyperscale_economizer.low:.1f} - {wue.hyperscale_economizer.high:.1f}",
        })
    
    return pd.DataFrame(rows)


def print_summary_table():
    """Print a formatted summary table like the image"""
    
    print("=" * 100)
    print("WUE RANGES BY CLIMATE TYPE (L/kWh)")
    print("Source: Lei & Masanet (2022), Resources, Conservation & Recycling")
    print("=" * 100)
    print()
    print(f"{'Climate Type':<20} | {'Traditional':<15} | {'Hyperscale':<15} | {'Air-Cooled':<12}")
    print(f"{'':<20} | {'Evaporative':<15} | {'w/Economizer':<15} | {'Only':<12}")
    print("-" * 20 + "-+-" + "-" * 15 + "-+-" + "-" * 15 + "-+-" + "-" * 12)
    
    for climate_type, wue in CLIMATE_TYPE_WUE.items():
        trad = f"{wue.traditional_evaporative.low:.1f} - {wue.traditional_evaporative.high:.1f}"
        hyper = f"{wue.hyperscale_economizer.low:.1f} - {wue.hyperscale_economizer.high:.1f}"
        air = f"< {wue.air_cooled.high:.2f}".replace("< 0.", "< .")
        
        print(f"{climate_type:<20} | {trad:<15} | {hyper:<15} | {air:<12}")
    
    print()
    print("=" * 100)
    print("WUE VALUES FOR 13 DATA CENTER LOCATIONS")
    print("=" * 100)
    print()
    print(f"{'Location':<25} | {'Zone':<6} | {'Climate Type':<12} | "
          f"{'Hyperscale (Low/Mid/High)':<28} | {'Trad Evap (Low/Mid/High)':<28}")
    print("-" * 25 + "-+-" + "-" * 6 + "-+-" + "-" * 12 + "-+-" + "-" * 28 + "-+-" + "-" * 28)
    
    for location, info in DATA_CENTER_LOCATIONS.items():
        ashrae_zone, climate_type, wue = get_wue_by_location(location)
        
        hyper = f"{wue.hyperscale_economizer.low:.2f} / {wue.hyperscale_economizer.mid:.2f} / {wue.hyperscale_economizer.high:.2f}"
        trad = f"{wue.traditional_evaporative.low:.2f} / {wue.traditional_evaporative.mid:.2f} / {wue.traditional_evaporative.high:.2f}"
        
        print(f"{location:<25} | {ashrae_zone:<6} | {climate_type:<12} | {hyper:<28} | {trad:<28}")
    
    print()


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    
    # Print formatted summary
    print_summary_table()
    
    # Generate and save CSV files
    print("Generating CSV files...")
    
    # 1. Climate type table
    df_climate = generate_climate_type_table()
    df_climate.to_csv("WUE_By_ClimateType.csv", index=False)
    print(f"  Saved: WUE_By_ClimateType.csv ({len(df_climate)} climate types)")
    
    # 2. Location table (main output for framework)
    df_locations = generate_location_wue_table()
    df_locations.to_csv("WUE_13_DataCenter_Locations.csv", index=False)
    print(f"  Saved: WUE_13_DataCenter_Locations.csv ({len(df_locations)} locations)")
    
    # 3. ASHRAE zone reference table
    df_ashrae = generate_ashrae_zone_table()
    df_ashrae.to_csv("ASHRAE_Zone_Reference.csv", index=False)
    print(f"  Saved: ASHRAE_Zone_Reference.csv ({len(df_ashrae)} zones)")
    
    print()
    print("=" * 100)
    print("METHODOLOGY DOCUMENTATION")
    print("=" * 100)
    print("""
WUE Value Derivation Method:

1. SOURCE: Lei & Masanet (2022) published WUE ranges by climate type and cooling technology

2. FOR RANGES (e.g., "1.0 - 1.8"):
   - WUE_low  = lower end of range (best practice)
   - WUE_high = upper end of range (poor practice)
   - WUE_mid  = (WUE_low + WUE_high) / 2 (central estimate)

3. FOR AIR-COOLED "< X" VALUES:
   - WUE_high = X (the stated upper bound)
   - WUE_mid  = X / 2
   - WUE_low  = X / 4

4. LOCATION MAPPING:
   - Each data center location mapped to ASHRAE climate zone
   - Zone determines climate type
   - Climate type determines WUE range
   
5. USAGE:
   - Use WUE_mid as central value in main results
   - Use WUE_low and WUE_high for sensitivity analysis
   
REFERENCE:
Lei, N., & Masanet, E. (2022). Climate- and technology-specific PUE and WUE 
estimations for U.S. data centers using a hybrid statistical and thermodynamics-
based approach. Resources, Conservation & Recycling, 182, 106323.
https://doi.org/10.1016/j.resconrec.2022.106323
""")
    
    print()
    print("Done!")